# DFL

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/12-dfl/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""DFL computes two-bin cross entropy and a differentiable expected distance."""
import torch
from torch import nn
from torch.nn import functional as F


def dfl(logits, target):
    left = target.floor().long()
    right = left + 1
    weight_right = target - left
    # The final bin cannot have a right neighbor; callers must validate the range.
    assert ((target >= 0) & (target < logits.shape[-1] - 1)).all()
    return ((1 - weight_right) * F.cross_entropy(logits, left, reduction='none')
            + weight_right * F.cross_entropy(logits, right, reduction='none')).mean()


def expected_distance(logits):
    return (logits.softmax(-1) * torch.arange(logits.shape[-1], dtype=logits.dtype)).sum(-1)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    target = torch.tensor([1.25])  # feature-cell units; ideal bin weights .75,.25
    logits = nn.Parameter(torch.zeros(1, 4))
    optimizer = torch.optim.SGD([logits], lr=.5)
    loss = dfl(logits, target)
    loss.backward()
    wanted_grad = torch.tensor([[.25, -.50, .0, .25]])
    assert torch.allclose(logits.grad, wanted_grad)
    print(f'uniform expectation={expected_distance(logits).item():.2f}; DFL={loss.item():.6f}')
    print('initial logit gradient:', logits.grad.tolist())
    optimizer.zero_grad()
    for _ in range(400):
        optimizer.zero_grad()
        dfl(logits, target).backward()
        optimizer.step()
    probs = logits.softmax(-1).detach()
    distance = expected_distance(logits).detach()
    assert abs(distance.item() - 1.25) < .02
    assert probs[0, 1] > .73 and probs[0, 2] > .24
    print('learned bin probabilities:', probs.round(decimals=4).tolist())
    print(f'learned expectation={distance.item():.4f} cells = {8 * distance.item():.4f} pixels at stride 8')
    # An expectation alone does not fix the distribution.
    a, b = torch.tensor([0., .75, .25, 0.]), torch.tensor([.375, 0., .625, 0.])
    assert torch.allclose((a * torch.arange(4)).sum(), (b * torch.arange(4)).sum())
    print('different distributions can share expectation=1.25')


if __name__ == '__main__':
    main()


uniform expectation=1.50; DFL=1.386294
initial logit gradient: [[0.25, -0.5, 0.0, 0.25]]
learned bin probabilities: [[0.0024999999441206455, 0.7475000023841858, 0.24740000069141388, 0.0024999999441206455]]
learned expectation=1.2500 cells = 9.9999 pixels at stride 8
different distributions can share expectation=1.25
